# 하네스와 일곱 기능

하네스가 프레임워크 위에서 어떤 문제를 해결하는지 분류합니다.

## 설치

고정된 패키지를 현재 커널에 설치합니다.

In [1]:
# 실습에서 검증한 패키지 버전을 현재 커널에 설치합니다.
%pip install -q deepagents==0.7.0 langchain==1.4.0 langchain-openai==1.6.0 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 실행 환경 준비

상위 폴더의 .env를 찾아 키 값은 출력하지 않고 버전만 확인합니다.


In [2]:
from importlib.metadata import version
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

# 현재 폴더부터 상위 폴더까지 .env를 찾고 값은 출력하지 않습니다.
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path)
print('deepagents', version('deepagents'))
print('langchain', version('langchain'))
assert version('deepagents') == '0.7.0'

deepagents 0.7.0
langchain 1.4.0


## 단계 1

설치한 0.7.0의 생성 함수 소스에서 일곱 기능이 조립되는 경로를 찾습니다.

### 코드 읽기와 비교 경계

하네스(harness)는 모델 하나를 뜻하지 않습니다. 모델 호출 주위에 도구, 파일 저장, 위임, 요약, 권한 같은 반복 기능을 조립해 긴 작업을 실행하는 틀입니다. 이 셀은 `inspect.getsource(create_deep_agent)`로 설치된 0.7.0의 생성 함수 소스를 문자열로 읽고, 각 기능을 조립하는 이름이 실제로 들어 있는지 확인합니다. `in source` 결과가 `True`라는 것은 생성 경로에 관련 코드가 존재한다는 뜻이며, 모든 실행에서 그 기능이 활성화되었다는 뜻은 아닙니다.

프레임워크와 하네스의 차이는 추상화 수준입니다. LangGraph는 상태와 노드의 실행 순서를 구성하고, LangChain의 에이전트는 모델과 도구 반복을 구성합니다. Deep Agents 하네스는 그 위에 장기 작업에서 반복되는 파일·서브에이전트·컨텍스트 관리 기능을 미리 조립합니다. 짧은 한 번의 도구 호출에는 하네스의 추가 표면이 필요하지 않을 수 있습니다. 기능 수를 비교할 때는 기본 활성 기능과 인자를 전달해야 생기는 선택 기능을 구분해야 합니다.


In [3]:
# 생성 함수가 조립하는 일곱 기능의 코드 경로를 확인합니다.
import inspect
import deepagents.graph as graph
source = inspect.getsource(graph.create_deep_agent)
features = {
    '파일시스템': 'FilesystemMiddleware' in source,
    '서브에이전트': 'SubAgentMiddleware' in source,
    '컨텍스트 요약': 'create_summarization_middleware' in source,
    '스킬': 'SkillsMiddleware' in source,
    '장기 메모리': 'MemoryMiddleware' in source,
    '권한·사람 승인': 'HumanInTheLoopMiddleware' in source,
    '비동기 서브에이전트': 'AsyncSubAgentMiddleware' in source,
}
for name, found in features.items(): print(name, found)

파일시스템 True
서브에이전트 True
컨텍스트 요약 True
스킬 True
장기 메모리 True
권한·사람 승인 True
비동기 서브에이전트 True


## 단계 2

찾은 일곱 항목이 모두 True인지와 합계가 7인지 확인합니다.

모든 값이 `True`라는 결과는 설치된 소스에 조립 경로가 있다는 뜻입니다. 각 기능이 실제 요청에서 선택되는지는 뒤 절의 실행 결과로 확인합니다.


In [4]:
# 일곱 기능의 조립 경로가 모두 확인됐는지 검사합니다.
print('확인한 기능 수:', sum(features.values()))
assert all(features.values())

확인한 기능 수: 7
